# Run Yolo

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # or your previous best.pt
model.train(
    data="floorplan.yaml",   # your train/val paths and classes
    epochs=100,
    lr0=0.0005,           # lower LR for small dataset
    patience=15,          # early stopping
    imgsz=640,
    batch=4,              # small batch for small GPU
    augment=True
)

# Predict validation dataset

In [ ]:
model = YOLO("runs/detect/train6/weights/best.pt")
model.predict(
    source=".. .. /Floorplan/dataset/images/val", # Add path here
    save=True,      # Saves the image with boxes drawn
    save_txt=True,  # Saves the coordinates in .txt files
    save_conf=True, # Optional: includes the confidence score in the txt
    imgsz=640       # Ensure this matches your training size
)

# Predict test dataset
### Label the spaces, and save both the objcts detected with confidence values and number of objects detected in each floorplan 

In [ ]:
import pandas as pd
import os
from ultralytics import YOLO

# 1. Load your model
model = YOLO("runs/detect/train6/weights/best.pt")

# 2. Run prediction
results = model.predict(
    source=". . /Floorplan/dataset/images/test", # Add path
    save=True,
    line_width=1,
    show_labels=False,
    show_conf=False
)

# Lists to hold our data
detail_data = []  # For floorplan_results.csv
summary_data = [] # For floorplan_counts.csv

# 3. Process the results
for r in results:
    img_name = os.path.basename(r.path)
    
    # Dictionary to count items in THIS specific image
    current_img_counts = {"Image": img_name}
    
    # Loop through every box detected in this image
    for box in r.boxes:
        class_id = int(box.cls[0])
        class_name = model.names[class_id]
        conf = float(box.conf[0])
        
        # Add to the Detail List
        detail_data.append({
            "Image": img_name,
            "Class Name": class_name,
            "Confidence": round(conf, 2)
        })
        
        # Add to the Summary Count
        current_img_counts[class_name] = current_img_counts.get(class_name, 0) + 1
    
    summary_data.append(current_img_counts)

# 4. Create and Save both DataFrames
df_details = pd.DataFrame(detail_data)
df_details.to_csv("floorplan_test-results.csv", index=False)

df_summary = pd.DataFrame(summary_data).fillna(0)
df_summary.to_csv("floorplan_test-counts.csv", index=False)

print("--- Done! ---")
print(f"Saved details for {len(detail_data)} total objects.")
print(f"Saved counts for {len(summary_data)} total images.")

# print("--- Summary of Detections ---")
# print(df_details)
# print(df_summary)